In [1]:
import pandas as pd
import numpy as np
import re
import nltk 

In [2]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [4]:
trueDf = pd.read_csv("../data/True.csv")
fakeDf = pd.read_csv("../data/Fake.csv")

In [5]:
trueDf.head(5)

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [6]:
fakeDf.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [7]:
trueDf["label"] = 1

In [8]:
fakeDf["label"] = 0

In [9]:
df = pd.concat([trueDf , fakeDf] , axis=0)

In [10]:
df = df.sample(frac=1 , random_state=42)

In [11]:
df.reset_index(drop=True , inplace=True)

In [12]:
df.head()

,title,text,subject,date,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",0
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",0
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",0
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",1
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",0


In [13]:
df = df[["text" , "label"]]

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    44898 non-null  object
 1   label   44898 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 701.7+ KB


In [15]:
df["label"].value_counts()

label
0    23481
1    21417
Name: count, dtype: int64

In [16]:
stop_words = set(stopwords.words('english'))

In [17]:
stemmer = PorterStemmer()

In [18]:
def cleaning_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+" , "" , text)
    text = re.sub(r"\d+" , "" ,text)
    text = re.sub(r"[^\w\s]" , "" , text)
    words = text.split()

    cleaned_words = []

    for word in words:
        if word not in stop_words:
            #stemmed_words = stemmer.stem(word)
            cleaned_words.append(word)
    return " ".join(cleaned_words)


In [19]:
sample = "THIS is a Fake news website!!!!!!!!!?///&"
print(cleaning_text(sample))

fake news website


In [20]:
#df = df.sample(5000, random_state=42)

In [21]:
df["clean_text"] = df["text"].apply(cleaning_text)

In [22]:
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=20000,
    stop_words="english"
)

In [23]:
x = vectorizer.fit_transform(df["clean_text"])

In [24]:
y = df["label"]

In [25]:
x_train,x_test,y_train,y_test = train_test_split(
    x,y,
    random_state=42,
    test_size=0.2
)

In [34]:
model = MultinomialNB()

In [35]:
model.fit(x_train , y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [28]:
y_pred = model.predict(x_test)
y_pred = y_pred.astype(int)
y_test = y_test.astype(int)

In [29]:
accuracy = accuracy_score(y_test ,y_pred)
print("Accuracy:" , accuracy)

Accuracy: 0.9534521158129176


In [30]:
print(classification_report(y_test , y_pred))

              precision    recall  f1-score   support

           0       0.96      0.95      0.96      4669
           1       0.95      0.95      0.95      4311

    accuracy                           0.95      8980
   macro avg       0.95      0.95      0.95      8980
weighted avg       0.95      0.95      0.95      8980



In [37]:
import joblib

joblib.dump(model , "../src/Fake_News_detection_model.pkl")
joblib.dump(vectorizer , "../src/tfidf_vectorizer.pkl")

['../src/tfidf_vectorizer.pkl']